# 09 - XGBoost challenger model

A gradient boosting challenger trained on the cleaned raw features, using the same train/test split as the scorecard. The aim is to measure how much predictive power the explainable scorecard gives up.

In [2]:
import sys
sys.path.append("..")

import pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier

train = pd.read_parquet("../data/processed/train.parquet")
test = pd.read_parquet("../data/processed/test.parquet")

FEATURES = [c for c in train.columns if c not in ["TARGET", "SK_ID_CURR"]]

# Text columns become categories, using the categories seen in training
for col in train[FEATURES].select_dtypes(exclude="number").columns:
    cats = pd.Categorical(train[col]).categories
    train[col] = pd.Categorical(train[col], categories=cats)
    test[col] = pd.Categorical(test[col], categories=cats)

# Validation set for early stopping, taken from the training data only
X_tr, X_val, y_tr, y_val = train_test_split(
    train[FEATURES], train["TARGET"], test_size=0.2,
    stratify=train["TARGET"], random_state=42,
)

xgb = XGBClassifier(
    n_estimators=2000, learning_rate=0.05, max_depth=4,
    subsample=0.8, colsample_bytree=0.8,
    tree_method="hist", enable_categorical=True,
    eval_metric="auc", early_stopping_rounds=100,
)
xgb.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=200)

auc = roc_auc_score(test["TARGET"], xgb.predict_proba(test[FEATURES])[:, 1])
print(f"\nBest number of trees: {xgb.best_iteration}")
print(f"XGBoost test AUC: {auc:.3f}, Gini: {2 * auc - 1:.3f}")
print("Scorecard test Gini: 0.490")

[0]	validation_0-auc:0.69404
[200]	validation_0-auc:0.75061
[400]	validation_0-auc:0.75443
[600]	validation_0-auc:0.75471
[663]	validation_0-auc:0.75491

Best number of trees: 563
XGBoost test AUC: 0.765, Gini: 0.531
Scorecard test Gini: 0.490


## Find out what drives the XGBoost model

In [3]:
importance = pd.Series(
    xgb.get_booster().get_score(importance_type="gain")
).sort_values(ascending=False)

scorecard_vars = ["EXT_SOURCE_3", "EXT_SOURCE_2", "EXT_SOURCE_1", "DAYS_EMPLOYED",
                  "OCCUPATION_TYPE", "ORGANIZATION_TYPE", "NAME_EDUCATION_TYPE",
                  "REGION_RATING_CLIENT_W_CITY", "AMT_GOODS_PRICE",
                  "CREDIT_GOODS_RATIO", "CREDIT_TERM", "OWN_CAR_AGE"]

top = importance.head(20).to_frame("gain")
top["share"] = top["gain"] / importance.sum()
top["in_scorecard"] = top.index.isin(scorecard_vars)
print(top.style.format({"gain": "{:.1f}", "share": "{:.1%}"}).to_string())

 gain share in_scorecard
EXT_SOURCE_2 68.7 8.7% True
EXT_SOURCE_3 66.9 8.5% True
NAME_EDUCATION_TYPE 28.9 3.7% True
CREDIT_GOODS_RATIO 26.5 3.3% True
EXT_SOURCE_1 25.4 3.2% True
FLAG_DOCUMENT_3 20.1 2.5% False
CREDIT_TERM 19.2 2.4% True
DEF_30_CNT_SOCIAL_CIRCLE 18.3 2.3% False
DAYS_EMPLOYED 17.3 2.2% True
REGION_RATING_CLIENT_W_CITY 15.1 1.9% True
FLAG_WORK_PHONE 15.0 1.9% False
REG_CITY_NOT_LIVE_CITY 13.6 1.7% False
ORGANIZATION_TYPE 13.4 1.7% True
OWN_CAR_AGE 13.3 1.7% True
AMT_GOODS_PRICE 13.2 1.7% True
DEF_60_CNT_SOCIAL_CIRCLE 13.2 1.7% False
NAME_CONTRACT_TYPE 13.2 1.7% False
OCCUPATION_TYPE 13.0 1.6% True
DAYS_BIRTH 12.3 1.5% False
AMT_ANNUITY 12.0 1.5% False



## Conclusion

The XGBoost challenger reaches a test Gini of 0.531, 0.041 above the scorecard (0.490), using about 75 variables instead of 12.

All 12 scorecard variables are among XGBoost's 20 most important features, and the top five largely coincide, confirming the scorecard's core risk drivers. XGBoost's extra power comes from many small contributions and interactions, plus variables excluded from the scorecard for good reasons: FLAG_DOCUMENT_3 (undocumented meaning), the social-circle default counts (fairness and data protection concerns) and age (wrong-sign coefficient in the linear model).

**Decision:** the scorecard remains the champion model. A gap of 0.04 Gini does not outweigh the scorecard's transparency, which IFRS 9 and supervisory expectations require. XGBoost serves as a benchmark, showing where the scorecard could be improved.